# Convolutional Neural Networks for Histopathology

_Notebook made by Zaid De Anda_

Today we will connect what we already saw about images, filters, and neural networks with a case much closer to medicine: classifying colorectal tissue patches.

The idea of this notebook is:

* remember why convolutions are useful
* train a small CNN on MNIST, just to warm up
* train a LeNet-like CNN on NCT-CRC-HE-100K
* evaluate with medical metrics, not only accuracy
* peek inside the kernels and activation maps
* use Grad-CAM to explain predictions a little bit
* close with a discussion about ethics in medical AI

Huge disclaimer: this is an educational exercise. It is not, and does not pretend to be, a clinical diagnostic tool.

## 1. Why CNNs?

Previously, we saw that a dense neural network can learn patterns. But with images there is a small problem: if we flatten an image, we lose spatial structure.

A CNN does something closer to what we did with kernels: it learns small filters that slide over the image. At first they may detect edges, textures, or color changes; later, more complex combinations.

In short: if the input is an image, it helps if the network knows that neighboring pixels matter.

In [ ]:
import importlib.util
import subprocess
import sys

for package in ["torch", "torchvision"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

In [ ]:
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.notebook import tqdm, trange

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 2. Mini demo: MNIST with a CNN

Before jumping into histopathology, let's make a quick stop at MNIST.

This is just a warm-up: we want to see the full cycle once: load images, train a CNN, and measure how well it predicts.

In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

mnist_train = datasets.MNIST("data", train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST("data", train=False, download=True, transform=mnist_transform)

mnist_train_small = Subset(mnist_train, range(5000))
mnist_test_small = Subset(mnist_test, range(1000))

mnist_train_loader = DataLoader(mnist_train_small, batch_size=64, shuffle=True)
mnist_test_loader = DataLoader(mnist_test_small, batch_size=256)

images, labels = next(iter(mnist_train_loader))
print("Batch shape:", images.shape)

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"label: {labels[i].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=0),
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.flatten_dim = 16 * 4 * 4

        self.fc = nn.Linear(self.flatten_dim, 120)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(120, 84)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(84, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.adaptive_pool(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        out = self.relu(out)
        out = self.fc1(out)
        out = self.relu1(out)
        out = self.fc2(out)
        return out


mnist_model = MNISTCNN().to(device)
mnist_loss_fn = nn.CrossEntropyLoss()
mnist_optimizer = torch.optim.Adam(mnist_model.parameters(), lr=0.001)

mnist_model.train()
for epoch in range(2):
    running_loss = 0.0
    for images, labels in mnist_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = mnist_model(images)
        loss = mnist_loss_fn(logits, labels)

        mnist_optimizer.zero_grad()
        loss.backward()
        mnist_optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch + 1} | loss = {running_loss / len(mnist_train_loader):.4f}")

In [ ]:
mnist_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in mnist_test_loader:
        images = images.to(device)
        labels = labels.to(device)
        predictions = mnist_model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"MNIST test accuracy: {correct / total:.3f}")

for evaluating this model, we use a metric called accuracy, which is _how many predictions were right_. this only works with balanced datasets

## 3. Medical dataset: NCT-CRC-HE-100K

Now, let's move to colorectal tissue images.

NCT-CRC-HE-100K contains histology image patches. The full dataset has 9 tissue classes, but for this class we will use only two:

* `NORM`: normal colon mucosa
* `TUM`: colorectal adenocarcinoma epithelium

This gives us a very clear question: given a tissue patch, can we distinguish normal tissue from tumor tissue?

For the medical metrics, we will treat `TUM` as the positive class and `NORM` as the negative class.

Careful: even though this dataset is excellent for learning, getting high accuracy here does not mean we have a reliable clinical tool.

### Downloading the dataset

We will use Kaggle through `kagglehub`, just like we would in Colab.

If you already have the dataset mounted in Drive, you can skip this cell and manually point `dataset_root` to the folder that contains the class folders.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("kagglehub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

import kagglehub

kaggle_path = Path(kagglehub.dataset_download("imrankhan77/nct-crc-he-100k"))
expected_classes = {"ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"}

dataset_root_candidates = [
    path for path in kaggle_path.rglob("*")
    if path.is_dir() and expected_classes.issubset({p.name for p in path.iterdir() if p.is_dir()})
]

if expected_classes.issubset({p.name for p in kaggle_path.iterdir() if p.is_dir()}):
    dataset_root_candidates.insert(0, kaggle_path)

if dataset_root_candidates:
    dataset_root = dataset_root_candidates[0]
else:
    raise FileNotFoundError("Could not find the NCT-CRC-HE-100K class folders.")

print("Dataset root:", dataset_root)

In [ ]:
image_size = 96

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

base_dataset = datasets.ImageFolder(dataset_root)
target_classes = ["NORM", "TUM"]
target_class_to_new_label = {
    base_dataset.class_to_idx[class_name]: new_label
    for new_label, class_name in enumerate(target_classes)
}

print("All classes:", base_dataset.classes)
print("Target classes:", target_classes)
print("Total images:", len(base_dataset))

In [ ]:
class BinaryLabelSubset(Subset):
    def __init__(self, dataset, indices, label_map):
        super().__init__(dataset, indices)
        self.label_map = label_map

    def __getitem__(self, idx):
        image, original_label = super().__getitem__(idx)
        return image, self.label_map[original_label]

    def __getitems__(self, indices):
        return [self.__getitem__(idx) for idx in indices]


# We use a balanced NORM/TUM subset so training stays fast during class.
max_images_per_class = 600

indices_by_class = {class_idx: [] for class_idx in target_class_to_new_label}
for index, (_, class_idx) in enumerate(base_dataset.samples):
    if class_idx in indices_by_class:
        indices_by_class[class_idx].append(index)

selected_indices = []
for class_idx, indices in indices_by_class.items():
    random.shuffle(indices)
    selected_indices.extend(indices[:max_images_per_class])

selected_labels = [target_class_to_new_label[base_dataset.samples[i][1]] for i in selected_indices]

train_valid_indices, test_indices = train_test_split(
    selected_indices,
    test_size=0.2,
    random_state=42,
    stratify=selected_labels,
)

train_valid_labels = [target_class_to_new_label[base_dataset.samples[i][1]] for i in train_valid_indices]

train_indices, valid_indices = train_test_split(
    train_valid_indices,
    test_size=0.2,
    random_state=42,
    stratify=train_valid_labels,
)

train_dataset_full = datasets.ImageFolder(dataset_root, transform=train_transform)
eval_dataset_full = datasets.ImageFolder(dataset_root, transform=eval_transform)

train_dataset = BinaryLabelSubset(train_dataset_full, train_indices, target_class_to_new_label)
valid_dataset = BinaryLabelSubset(eval_dataset_full, valid_indices, target_class_to_new_label)
test_dataset = BinaryLabelSubset(eval_dataset_full, test_indices, target_class_to_new_label)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=64, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, num_workers=2)

print("Train images:", len(train_dataset))
print("Valid images:", len(valid_dataset))
print("Test images:", len(test_dataset))

In [ ]:
def denormalize(image_tensor):
    image_tensor = image_tensor * 0.5 + 0.5
    return image_tensor.clamp(0, 1)


images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    image = denormalize(images[i]).permute(1, 2, 0)
    plt.imshow(image)
    plt.title(target_classes[labels[i].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Let's train a CNN with NCT-CRC-HE-100K

For this notebook, we will use a LeNet5-inspired network.

We are not trying to build the best possible model; we are trying to understand the whole workflow: images, convolutions, training, metrics, and interpretability.

In [ ]:
class HistologyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.flatten_dim = 64 * 4 * 4

        self.fc = nn.Linear(self.flatten_dim, 120)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(120, 84)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(84, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.adaptive_pool(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        out = self.relu(out)
        out = self.fc1(out)
        out = self.relu1(out)
        out = self.fc2(out)
        return out


model = HistologyCNN(num_classes=len(target_classes)).to(device)
print(model)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", trainable_params)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc


def train(model, iterator, optimizer, criterion, device):
    epoch_loss = 0
    epoch_acc = 0

    model.train()

    for (x, y) in tqdm(iterator, desc="Training", leave=False):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred = model(x)
        loss = criterion(y_pred, y)
        acc = calculate_accuracy(y_pred, y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


def evaluate(model, iterator, criterion, device):
    epoch_loss = 0
    epoch_acc = 0

    model.eval()

    with torch.no_grad():
        for (x, y) in tqdm(iterator, desc="Evaluating", leave=False):
            x = x.to(device)
            y = y.to(device)

            y_pred = model(x)
            loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)


def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs


def predict(model, data_loader):
    model.eval()
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            logits = model(images)
            probabilities = torch.softmax(logits, dim=1)
            predictions = probabilities.argmax(dim=1)

            all_labels.extend(labels.numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    return np.array(all_labels), np.array(all_predictions), np.array(all_probabilities)

In [ ]:
EPOCHS = 3
train_losses = []
valid_losses = []

best_valid_loss = float("inf")
model_path = "histology-cnn.pt"

for epoch in trange(EPOCHS):
    start_time = time.monotonic()

    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), model_path)

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f"Epoch: {epoch + 1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s")
    print(f"\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc * 100:.2f}%")
    print(f"\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc * 100:.2f}%")

model.load_state_dict(torch.load(model_path, map_location=device))
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.3f} | Test Acc: {test_acc * 100:.2f}%")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_losses, marker="o", label="train")
plt.plot(valid_losses, marker="o", label="valid")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("NCT-CRC-HE-100K loss")
plt.legend()
plt.show()

## 5. Medical metrics for binary classification

In medical problems, we can't rely only on accuracy.

Let's imagine we have a model that predicts whether you are going to have a car crash today. An average person has around 3-4 vehicle accidents over their lifetime. If we say someone drives for 55 years, that is 55 * 365 = 20,075 possible driving days, and only a few of those days involve a crash. If a computer program always says "You are not going to crash today", it would be right most of the time! Is this a good model?

So we need other metrics.

We can start with precision. If "Tumor" is the positive class and "Normal" is the negative class, **precision** asks: out of all cases predicted as tumor, how many were actually tumor?

$$
precision = \frac{TP}{TP+FP}
$$

TP means true positive and FP means false positive.

Another useful metric is recall, also called sensitivity. This is the ratio of positive instances that are correctly detected by the classifier: out of all tumors, how many did we catch?

$$
recall = \frac{TP}{TP+FN}
$$

TP means true positive and FN means false negative.

We can also combine both recall and precision to make the F1 score, which is calculated by:

$$
F_1 = \frac{2}{\frac{1}{precision} + \frac{1}{recall}}
$$

Which metric we want to prioritize really depends on the case.

Question for class: which error worries us more, a false positive or a false negative?

In [ ]:
y_true, y_pred, y_prob = predict(model, test_loader)

tum_label = target_classes.index("TUM")
norm_label = target_classes.index("NORM")
tum_probabilities = y_prob[:, tum_label]

accuracy = accuracy_score(y_true, y_pred)
sensitivity = recall_score(y_true, y_pred, pos_label=tum_label, zero_division=0)
specificity = recall_score(y_true, y_pred, pos_label=norm_label, zero_division=0)
precision = precision_score(y_true, y_pred, pos_label=tum_label, zero_division=0)
f1 = f1_score(y_true, y_pred, pos_label=tum_label, zero_division=0)
roc_auc = roc_auc_score((y_true == tum_label).astype(int), tum_probabilities)

print(f"Accuracy:    {accuracy:.3f}")
print(f"Sensitivity: {sensitivity:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"Precision:   {precision:.3f}")
print(f"F1 score:    {f1:.3f}")
print(f"ROC-AUC:     {roc_auc:.3f}")

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[norm_label, tum_label]).ravel()
print(f"\nFalse positives: {fp}")
print(f"False negatives: {fn}")

We can also use the confusion matrix, which allows us to see how many:

* True positives: how many tumor samples did we find?
* False positives: how many normal samples did we identify as tumor?
* True negatives: how many normal samples did we identify as normal?
* False negatives: how many tumor samples did we miss?

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(cm, display_labels=target_classes).plot(
    cmap="Blues",
    ax=ax,
    xticks_rotation=45,
)
plt.title("Confusion matrix")
plt.show()

And last but not least, we can use the receiver operating characteristic curve, or ROC curve for short. It shows how sensitivity and false positive rate change as we move the decision threshold.

Specificity is the ratio of negative instances correctly classified as negative: how often we avoid false alarms.

In [ ]:
RocCurveDisplay.from_predictions(
    (y_true == tum_label).astype(int),
    tum_probabilities,
    name="TUM",
)
plt.title("ROC curve")
plt.show()

# Peeking inside the model

Remember when we trained kernels to discover good filters?

Now let's see what our first convolutional layer learned and what activations it produces on a real image from the dataset.

In [ ]:
first_kernels = model.layer1[0].weight.detach().cpu()

plt.figure(figsize=(12, 5))
for i in range(min(8, first_kernels.shape[0])):
    kernel = first_kernels[i].mean(dim=0).numpy()
    kernel = (kernel - kernel.min()) / (kernel.max() - kernel.min() + 1e-8)

    plt.subplot(2, 4, i + 1)
    plt.imshow(kernel, cmap="gray")
    plt.title(f"kernel {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

These are some kernels from the first layer. Since our images are RGB, each kernel has 3 channels, but here we average those channels so we can visualize each kernel as a grayscale filter.

Now let's take one image and see the activation maps after the first convolutional block.

In [ ]:
images, labels = next(iter(test_loader))
img = images[0].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    first_activations = model.layer1(img).cpu()

plt.figure(figsize=(3, 3))
plt.imshow(denormalize(images[0]).permute(1, 2, 0))
plt.title(f"Original: {target_classes[labels[0].item()]}")
plt.axis("off")
plt.show()

plt.figure(figsize=(12, 5))
for i in range(min(8, first_activations.shape[1])):
    plt.subplot(2, 4, i + 1)
    plt.imshow(first_activations[0, i], cmap="gray")
    plt.title(f"map {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Grad-CAM

Kernels and activation maps let us see internal parts of the network, but we want something more connected to the final prediction.

Grad-CAM helps us highlight image regions that strongly influenced a class prediction. It does not mean the network thinks like a pathologist, but it gives us a useful window for inspection.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self.save_activations)
        self.backward_handle = target_layer.register_full_backward_hook(self.save_gradients)

    def save_activations(self, module, inputs, output):
        self.activations = output

    def save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, image_tensor, class_idx):
        self.model.zero_grad()
        logits = self.model(image_tensor)
        score = logits[:, class_idx].sum()
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image_tensor.shape[-2:], mode="bilinear", align_corners=False)

        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def close(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

In [ ]:
def show_gradcam(model, dataset, index, class_idx=None):
    model.eval()
    image, label = dataset[index]
    image_batch = image.unsqueeze(0).to(device)

    logits = model(image_batch)
    probabilities = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
    predicted_class = int(probabilities.argmax())

    if class_idx is None:
        class_idx = predicted_class

    gradcam = GradCAM(model, model.layer3[0])
    cam = gradcam(image_batch, class_idx)
    gradcam.close()

    image_to_show = denormalize(image).permute(1, 2, 0).numpy()

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(image_to_show)
    plt.title(f"True: {target_classes[label]}")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cam, cmap="jet")
    plt.title("Grad-CAM heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(image_to_show)
    plt.imshow(cam, cmap="jet", alpha=0.45)
    plt.title(
        f"Pred: {target_classes[predicted_class]}\n"
        f"Confidence: {probabilities[predicted_class]:.2f}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Change the indices and compare correct predictions against mistakes.
show_gradcam(model, test_dataset, index=0)
show_gradcam(model, test_dataset, index=1)

## Try your own image

If you are running this notebook in Google Colab, you can upload an image and inspect the model prediction with Grad-CAM.

For the LeNet-like model, we will use the same `eval_transform` used by the NCT-CRC-HE-100K test set.

In [ ]:
def upload_image_from_colab():
    import io
    from PIL import Image

    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("This helper only works inside Google Colab.") from exc

    uploaded = files.upload()
    file_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[file_name])).convert("RGB")
    return image, file_name


def predict_uploaded_image_with_gradcam(model, pil_image, transform, target_layer, title):
    model.eval()

    image_tensor = transform(pil_image).unsqueeze(0).to(device)
    image_tensor.requires_grad_(True)

    logits = model(image_tensor)
    probabilities = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
    predicted_class = int(probabilities.argmax())

    gradcam = GradCAM(model, target_layer)
    cam = gradcam(image_tensor, predicted_class)
    gradcam.close()

    display_image = pil_image.resize((image_tensor.shape[-1], image_tensor.shape[-2]))

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(display_image)
    plt.title("Uploaded image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cam, cmap="jet")
    plt.title("Grad-CAM heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(display_image)
    plt.imshow(cam, cmap="jet", alpha=0.45)
    plt.title(
        f"{title}\n"
        f"Pred: {target_classes[predicted_class]} | "
        f"Conf: {probabilities[predicted_class]:.2f}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    for class_name, probability in zip(target_classes, probabilities):
        print(f"{class_name}: {probability:.4f}")

In [ ]:
uploaded_image, uploaded_name = upload_image_from_colab()

predict_uploaded_image_with_gradcam(
    model=model,
    pil_image=uploaded_image,
    transform=eval_transform,
    target_layer=model.layer3[0],
    title=f"LeNet-like CNN: {uploaded_name}",
)

## Final challenges

I challenge you, student! Change one thing and observe what happens:

* train with more or fewer images per class
* increase the number of kernels
* change the learning rate
* add dropout
* inspect Grad-CAM on an incorrect prediction
* compare accuracy against sensitivity and specificity
* move the decision threshold and observe how false positives / false negatives change

Do not only ask: did accuracy improve?

Also ask: would this change make the model safer or more trustworthy in a medical context?

## Final final option: ResNet50 on all NORM/TUM images

If you still have time, GPU, and patience, here is a more practical baseline: transfer learning with ResNet50.

This version does **not** use the small class subset. It is still restricted to the two selected classes, `NORM` and `TUM`, but it uses all available images from those two classes.

This can take significantly longer than the didactic LeNet-like model.

In [ ]:
from torchvision import models

resnet_weights = models.ResNet50_Weights.DEFAULT
resnet_preprocess = resnet_weights.transforms()

resnet_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=resnet_preprocess.mean,
        std=resnet_preprocess.std,
    ),
])

resnet_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=resnet_preprocess.mean,
        std=resnet_preprocess.std,
    ),
])

In [ ]:
# All available NORM/TUM images, no max_images_per_class.
full_indices = []
full_labels = []

for index, (_, class_idx) in enumerate(base_dataset.samples):
    if class_idx in target_class_to_new_label:
        full_indices.append(index)
        full_labels.append(target_class_to_new_label[class_idx])

resnet_train_valid_indices, resnet_test_indices = train_test_split(
    full_indices,
    test_size=0.2,
    random_state=42,
    stratify=full_labels,
)

resnet_train_valid_labels = [
    target_class_to_new_label[base_dataset.samples[i][1]]
    for i in resnet_train_valid_indices
]

resnet_train_indices, resnet_valid_indices = train_test_split(
    resnet_train_valid_indices,
    test_size=0.2,
    random_state=42,
    stratify=resnet_train_valid_labels,
)

resnet_train_full = datasets.ImageFolder(dataset_root, transform=resnet_train_transform)
resnet_eval_full = datasets.ImageFolder(dataset_root, transform=resnet_eval_transform)

resnet_train_dataset = BinaryLabelSubset(resnet_train_full, resnet_train_indices, target_class_to_new_label)
resnet_valid_dataset = BinaryLabelSubset(resnet_eval_full, resnet_valid_indices, target_class_to_new_label)
resnet_test_dataset = BinaryLabelSubset(resnet_eval_full, resnet_test_indices, target_class_to_new_label)

resnet_train_loader = DataLoader(resnet_train_dataset, batch_size=32, shuffle=True, num_workers=2)
resnet_valid_loader = DataLoader(resnet_valid_dataset, batch_size=64, num_workers=2)
resnet_test_loader = DataLoader(resnet_test_dataset, batch_size=64, num_workers=2)

print("ResNet50 train images:", len(resnet_train_dataset))
print("ResNet50 valid images:", len(resnet_valid_dataset))
print("ResNet50 test images:", len(resnet_test_dataset))

In [ ]:
resnet_model = models.resnet50(weights=resnet_weights)

# Freeze the pretrained feature extractor first.
for param in resnet_model.parameters():
    param.requires_grad = False

resnet_model.fc = nn.Linear(resnet_model.fc.in_features, len(target_classes))
resnet_model = resnet_model.to(device)

resnet_trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(resnet_model)
print("Trainable parameters:", resnet_trainable_params)

In [ ]:
resnet_criterion = nn.CrossEntropyLoss()
resnet_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=0.001,
)

RESNET_EPOCHS = 3
resnet_train_losses = []
resnet_valid_losses = []

best_resnet_valid_loss = float("inf")
resnet_model_path = "histology-resnet50.pt"

for epoch in trange(RESNET_EPOCHS):
    start_time = time.monotonic()

    train_loss, train_acc = train(
        resnet_model,
        resnet_train_loader,
        resnet_optimizer,
        resnet_criterion,
        device,
    )
    valid_loss, valid_acc = evaluate(
        resnet_model,
        resnet_valid_loader,
        resnet_criterion,
        device,
    )

    if valid_loss < best_resnet_valid_loss:
        best_resnet_valid_loss = valid_loss
        torch.save(resnet_model.state_dict(), resnet_model_path)

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    resnet_train_losses.append(train_loss)
    resnet_valid_losses.append(valid_loss)

    print(f"Epoch: {epoch + 1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s")
    print(f"\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc * 100:.2f}%")
    print(f"\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc * 100:.2f}%")

resnet_model.load_state_dict(torch.load(resnet_model_path, map_location=device))
resnet_test_loss, resnet_test_acc = evaluate(resnet_model, resnet_test_loader, resnet_criterion, device)

print(f"ResNet50 Test Loss: {resnet_test_loss:.3f} | Test Acc: {resnet_test_acc * 100:.2f}%")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(resnet_train_losses, marker="o", label="train")
plt.plot(resnet_valid_losses, marker="o", label="valid")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("ResNet50 loss on all NORM/TUM images")
plt.legend()
plt.show()

In [ ]:
resnet_y_true, resnet_y_pred, resnet_y_prob = predict(resnet_model, resnet_test_loader)
resnet_tum_probabilities = resnet_y_prob[:, tum_label]

resnet_accuracy = accuracy_score(resnet_y_true, resnet_y_pred)
resnet_sensitivity = recall_score(resnet_y_true, resnet_y_pred, pos_label=tum_label, zero_division=0)
resnet_specificity = recall_score(resnet_y_true, resnet_y_pred, pos_label=norm_label, zero_division=0)
resnet_precision = precision_score(resnet_y_true, resnet_y_pred, pos_label=tum_label, zero_division=0)
resnet_f1 = f1_score(resnet_y_true, resnet_y_pred, pos_label=tum_label, zero_division=0)
resnet_roc_auc = roc_auc_score((resnet_y_true == tum_label).astype(int), resnet_tum_probabilities)

print(f"ResNet50 accuracy:    {resnet_accuracy:.3f}")
print(f"ResNet50 sensitivity: {resnet_sensitivity:.3f}")
print(f"ResNet50 specificity: {resnet_specificity:.3f}")
print(f"ResNet50 precision:   {resnet_precision:.3f}")
print(f"ResNet50 F1 score:    {resnet_f1:.3f}")
print(f"ResNet50 ROC-AUC:     {resnet_roc_auc:.3f}")

resnet_cm = confusion_matrix(resnet_y_true, resnet_y_pred)
ConfusionMatrixDisplay(resnet_cm, display_labels=target_classes).plot(cmap="Blues")
plt.title("ResNet50 confusion matrix")
plt.show()

### Try your own image with ResNet50

This repeats the upload experiment, but now with the ResNet50 model and its ImageNet preprocessing.

In [ ]:
uploaded_image, uploaded_name = upload_image_from_colab()

predict_uploaded_image_with_gradcam(
    model=resnet_model,
    pil_image=uploaded_image,
    transform=resnet_eval_transform,
    target_layer=resnet_model.layer4[-1],
    title=f"ResNet50: {uploaded_name}",
)